In [ ]:
import zarr
import numpy as np
import polars as pl
from anngeno import AnnGeno

## Sanity check AnnGeno

In [ ]:
vm = pl.read_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/variant_metadata.parquet")
vm

In [ ]:
an = pl.read_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/annotations.parquet")
an['region'].unique()

In [ ]:
ag = AnnGeno(
    "/home/dnanexus/data_dir/dms_anngeno.ag",
    low_mem = True
)
ag

In [ ]:
gene_id = 'ENSG00000106633' # GCK
reg_dict = ag.get_region(gene_id)

geno = reg_dict['genotypes']
anno_df = reg_dict['annotations']
geno

In [ ]:
mac = np.load("/home/dnanexus/data_dir/all_mac.npy")
gene_mac = mac[anno_df['col']]
gene_mac

In [ ]:
geno_sum = geno.sum(axis=1)
geno_sum

In [ ]:
# Boolean mask where values differ
diff_mask = geno_sum != gene_mac

# Indices where they differ
diff_positions = np.where(diff_mask)[0]
diff_positions

## Create new subsetted AnnGeno

### Steps

1. first run `python subset_anngeno.py`
2. then create a dir `new.ag`, this will be the new anngeno
3. move the `new_genotypes/` into `new.ag/zarr_store/genotypes`
4. add the `samples/` zarr array to `new.ag/zarr_store/samples` as well
5. create `new.ag/zarr_store/zarr.json` and `new.ag/zarr_store/samples/zarr.json`

The code below will create the files:
- `new.ag/variant_metadata.parquet`
- `new.ag/annotations.parquet`

### Create metadata and annotations parquets

In [ ]:
an = pl.scan_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/annotations.parquet")
# an = pl.scan_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")

coding_annos = an.filter((pl.col("relative_cds_position_is_nan") == 0)| (pl.col("loftee_hc_is_nan") == 0)).collect()

coding_vars = coding_annos['id'].unique().to_list()
len(coding_vars)

In [ ]:
geno = zarr.open("/home/dnanexus/data_dir/dms_coding.ag/zarr_store/genotypes", mode='r')
geno.shape

In [ ]:
# Correctly sorted variant metadata
vm_og = pl.read_parquet('/home/dnanexus/data_dir/dms_anngeno.ag/variant_metadata.parquet')

vm = vm_og.filter(pl.col('id').is_in(coding_vars))
vm = vm.with_columns(pl.arange(0, pl.len(), eager=False).alias("col"))
vm

In [ ]:
coding_annos = vm[['id', 'col']].join(coding_annos.drop('col'), on='id')
coding_annos

In [ ]:
vm.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/variant_metadata.parquet")
coding_annos.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")

### Sanity check new anngeno

In [ ]:
vm = pl.read_parquet("/home/dnanexus/data_dir/dms_coding.ag/variant_metadata.parquet")
vm

In [ ]:
an = pl.read_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")
an['region'].unique()

In [ ]:
ag = AnnGeno(
    "/home/dnanexus/data_dir/dms_coding.ag",
    low_mem = True
)
ag

In [ ]:
gene_id = 'ENSG00000106633' # GCK
reg_dict = ag.get_region(gene_id)

geno = reg_dict['genotypes']
anno_df = reg_dict['annotations']
geno

In [ ]:
geno_sum = geno.sum(axis=1)
geno_maf = (geno_sum/(geno.shape[1]*2)).astype(np.float32)
geno_maf[:10]

In [ ]:
diff_positions = np.where(anno_df["maf"].to_numpy() != geno_maf)[0]
diff_positions